# 10. k-Nearest Neighbors (kNN) and Naive Bayes

## Learning Objectives
- Understand the distance-based classification principle of kNN
- Learn distance metrics (Euclidean, Manhattan, Minkowski)
- Master optimal k value selection methods
- Understand probability-based classification with Naive Bayes
- Compare Gaussian, Multinomial, and Bernoulli NB
- Apply to text classification

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.datasets import (
    load_iris, load_breast_cancer, load_diabetes, load_digits,
    make_classification, fetch_20newsgroups
)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from scipy.spatial.distance import euclidean, cityblock, minkowski, chebyshev
from time import time

# Font settings
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(42)

## 1. k-Nearest Neighbors (kNN) Concept

kNN is a Lazy Learning algorithm.

**How it works**:
1. When new data arrives
2. Find the k closest neighbors from the training data
3. Predict by majority vote (classification) or averaging (regression)

**Characteristics**:
- No model is built during training (all data is stored)
- Non-parametric method (no assumption about data distribution)
- Slow prediction time

In [ ]:
# 2D data for kNN visualization
X, y = make_classification(
    n_samples=100, n_features=2, n_redundant=0,
    n_informative=2, n_clusters_per_class=1, random_state=42
)

# Compare multiple k values
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
k_values = [1, 5, 15]

for ax, k in zip(axes, k_values):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X, y)

    # Decision boundary
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))

    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='black')
    ax.set_title(f'k = {k}\nAccuracy = {knn.score(X, y):.3f}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

## 2. Basic kNN Usage

In [ ]:
# Load data
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

# Why: kNN stores the entire training set and computes distances at prediction time.
# weights='uniform' gives equal vote to all k neighbors; 'distance' weights by
# 1/d so closer neighbors have more influence — often better for overlapping classes.
knn = KNeighborsClassifier(
    n_neighbors=5,           # k value
    weights='uniform',       # weights: 'uniform' or 'distance'
    algorithm='auto',        # algorithm: 'auto', 'ball_tree', 'kd_tree', 'brute'
    metric='minkowski',      # distance metric: 'euclidean', 'manhattan', 'minkowski'
    p=2                      # minkowski p value (2=euclidean, 1=manhattan)
)

knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

print("kNN Classification Results:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

## 3. Distance Metrics

The core of kNN is distance computation.

Key distance metrics:
- **Euclidean (L2)**: d = sqrt(sum((xi - yi)^2))
- **Manhattan (L1)**: d = sum(|xi - yi|)
- **Minkowski**: d = (sum(|xi - yi|^p))^(1/p)
- **Chebyshev (L-infinity)**: d = max(|xi - yi|)

In [ ]:
# Distance metric examples
point1 = np.array([1, 2, 3])
point2 = np.array([4, 5, 6])

print("Distance Metric Examples:")
print(f"  Point 1: {point1}")
print(f"  Point 2: {point2}")
print()
print(f"  Euclidean distance:    {euclidean(point1, point2):.4f}")
print(f"  Manhattan distance:    {cityblock(point1, point2):.4f}")
print(f"  Minkowski (p=3):       {minkowski(point1, point2, p=3):.4f}")
print(f"  Chebyshev distance:    {chebyshev(point1, point2):.4f}")

In [ ]:
# Performance comparison by distance metric
metrics = ['euclidean', 'manhattan', 'chebyshev']

print("Performance by Distance Metric (Iris):")
print("-" * 40)
for metric in metrics:
    knn = KNeighborsClassifier(n_neighbors=5, metric=metric)
    knn.fit(X_train, y_train)
    acc = knn.score(X_test, y_test)
    print(f"  {metric:12s}: {acc:.4f}")

## 4. Optimal k Value Selection

If k is too small, overfitting occurs; if too large, underfitting occurs.
We find the optimal k using cross-validation.

In [ ]:
# Performance change with different k values
k_range = range(1, 31)
train_scores = []
test_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_scores.append(knn.score(X_train, y_train))
    test_scores.append(knn.score(X_test, y_test))

# Visualization
plt.figure(figsize=(10, 6))
plt.plot(k_range, train_scores, 'o-', label='Train')
plt.plot(k_range, test_scores, 's-', label='Test')
plt.xlabel('k (Number of Neighbors)')
plt.ylabel('Accuracy')
plt.title('kNN: k vs Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(k_range[::2])
plt.tight_layout()
plt.show()

# Find optimal k
best_k = k_range[np.argmax(test_scores)]
print(f"Optimal k: {best_k}")
print(f"Best test accuracy: {max(test_scores):.4f}")

In [ ]:
# k selection via cross-validation
k_range = range(1, 31)
cv_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

# Visualization
plt.figure(figsize=(10, 6))
plt.plot(k_range, cv_scores, 'o-', color='green')
plt.xlabel('k')
plt.ylabel('Cross-Validation Accuracy')
plt.title('kNN: k Selection with 5-Fold Cross-Validation')
plt.grid(True, alpha=0.3)
plt.xticks(k_range[::2])
plt.tight_layout()
plt.show()

best_k_cv = k_range[np.argmax(cv_scores)]
print(f"Cross-validation optimal k: {best_k_cv}")
print(f"Best CV accuracy: {max(cv_scores):.4f}")

## 5. Weighted kNN

Adjusts neighbor weights based on distance.

- **uniform**: Equal weight for all neighbors
- **distance**: Higher weight for closer neighbors (weight = 1/distance)

In [ ]:
# Weight method comparison
weights = ['uniform', 'distance']

print("Weight Method Comparison:")
print("-" * 40)
for weight in weights:
    knn = KNeighborsClassifier(n_neighbors=5, weights=weight)
    knn.fit(X_train, y_train)
    acc = knn.score(X_test, y_test)
    print(f"  {weight:10s}: {acc:.4f}")

In [ ]:
# Distance-weighted kNN visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, weight in zip(axes, weights):
    knn = KNeighborsClassifier(n_neighbors=15, weights=weight)
    knn.fit(X[:, :2], y)

    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))

    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='black')
    ax.set_title(f'weights = {weight}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

## 6. kNN Regression

kNN can also be used for regression problems.
It predicts by averaging the k nearest neighbors.

In [ ]:
# Load data
diabetes = load_diabetes()
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    diabetes.data, diabetes.target, test_size=0.2, random_state=42
)

# Why: kNN is distance-based, so features on different scales would distort
# neighbor selection. StandardScaler ensures all features contribute equally.
# fit on train only, transform on test — prevents data leakage.
scaler = StandardScaler()
X_train_d_scaled = scaler.fit_transform(X_train_d)
X_test_d_scaled = scaler.transform(X_test_d)

# kNN Regression
knn_reg = KNeighborsRegressor(n_neighbors=5, weights='distance')
knn_reg.fit(X_train_d_scaled, y_train_d)
y_pred_d = knn_reg.predict(X_test_d_scaled)

print("kNN Regression Results:")
print(f"  MSE: {mean_squared_error(y_test_d, y_pred_d):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test_d, y_pred_d)):.4f}")
print(f"  R2: {r2_score(y_test_d, y_pred_d):.4f}")

## 7. kNN Algorithm Comparison

For large datasets, the choice of search algorithm is important.

- **brute**: Exhaustive search (O(n))
- **kd_tree**: Uses KD-Tree (efficient for low dimensions)
- **ball_tree**: Uses Ball-Tree (efficient for high dimensions)

In [ ]:
# Time comparison by algorithm
algorithms = ['brute', 'kd_tree', 'ball_tree']

print("Time Comparison by Algorithm:")
print("-" * 60)
for algo in algorithms:
    knn = KNeighborsClassifier(n_neighbors=5, algorithm=algo)

    # Fit time
    start = time()
    knn.fit(X_train, y_train)
    fit_time = time() - start

    # Prediction time
    start = time()
    knn.predict(X_test)
    pred_time = time() - start

    print(f"  {algo:10s}: fit={fit_time:.4f}s, predict={pred_time:.4f}s")

## 8. Naive Bayes

### Bayes' Theorem

**P(y|X) = P(X|y) x P(y) / P(X)**

- P(y|X): Posterior probability (class probability given features)
- P(X|y): Likelihood (feature probability given class)
- P(y): Prior probability (base probability of class)
- P(X): Evidence (probability of features)

### Naive Assumption

Assumes all features are independent of each other:
**P(X|y) = P(x1|y) x P(x2|y) x ... x P(xn|y)**

## 9. Gaussian Naive Bayes

Assumes continuous features follow a Gaussian (normal) distribution.
**P(xi|y) = N(xi; mu_y, sigma_y)**

In [ ]:
# Gaussian Naive Bayes
gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred_nb = gnb.predict(X_test)

print("Gaussian Naive Bayes Results:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}")

# Check learned parameters
print(f"\nClass prior probabilities: {gnb.class_prior_}")
print(f"\nClass means (first 2 features):")
print(gnb.theta_[:, :2])
print(f"\nClass variances (first 2 features):")
print(gnb.var_[:, :2])

In [ ]:
# Probability prediction
y_proba = gnb.predict_proba(X_test[:5])

print("Probability Predictions (first 5):")
print(f"Classes: {iris.target_names}")
print(y_proba)
print(f"\nPredicted classes: {gnb.predict(X_test[:5])}")
print(f"Actual classes: {y_test[:5]}")

## 10. Multinomial Naive Bayes - Text Classification

Used for discrete/count features, primarily for text classification (word frequencies).

**P(xi|y) = (Nyi + alpha) / (Ny + alpha*n)**

- alpha: Laplace smoothing parameter (solves the zero frequency problem)

In [ ]:
# Load news data
categories = ['sci.space', 'rec.sport.baseball', 'talk.politics.misc']
newsgroups = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

print(f"News data: {len(newsgroups.data)} articles")
print(f"Categories: {categories}")
print(f"\nFirst article (excerpt):\n{newsgroups.data[0][:200]}...")

In [ ]:
# Text vectorization
vectorizer = CountVectorizer(max_features=5000, stop_words='english')
X_news = vectorizer.fit_transform(newsgroups.data)
y_news = newsgroups.target

print(f"Vector shape: {X_news.shape}")
print(f"Number of features: {len(vectorizer.get_feature_names_out())}")

# Train/test split
X_train_news, X_test_news, y_train_news, y_test_news = train_test_split(
    X_news, y_news, test_size=0.2, random_state=42
)

In [ ]:
# Why: MultinomialNB models P(word|class) using word counts, making it natural
# for text. alpha=1.0 (Laplace smoothing) adds a pseudo-count to every word,
# preventing zero probabilities for words unseen in a class during training.
mnb = MultinomialNB(alpha=1.0)  # alpha: Laplace smoothing
mnb.fit(X_train_news, y_train_news)

y_pred_news = mnb.predict(X_test_news)

print("Multinomial Naive Bayes (Text Classification) Results:")
print(f"  Accuracy: {mnb.score(X_test_news, y_test_news):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_news, y_pred_news, target_names=categories))

In [ ]:
# Top words for each class
feature_names = vectorizer.get_feature_names_out()

print("Top 10 Words per Class:")
print("=" * 60)
for i, category in enumerate(categories):
    top_indices = mnb.feature_log_prob_[i].argsort()[-10:][::-1]
    top_words = [feature_names[idx] for idx in top_indices]
    print(f"\n{category}:")
    print(f"  {', '.join(top_words)}")

## 11. Bernoulli Naive Bayes

Used for binary features (0/1), classifies text based on word presence/absence.

In [ ]:
# Binary vectorization (word presence only)
binary_vectorizer = CountVectorizer(max_features=5000, binary=True, stop_words='english')
X_binary = binary_vectorizer.fit_transform(newsgroups.data)

X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_binary, y_news, test_size=0.2, random_state=42
)

# Bernoulli Naive Bayes
bnb = BernoulliNB(alpha=1.0)
bnb.fit(X_train_bin, y_train_bin)

print("Bernoulli Naive Bayes Results:")
print(f"  Accuracy: {bnb.score(X_test_bin, y_test_bin):.4f}")

## 12. Naive Bayes Model Comparison

In [ ]:
# Digit image data
digits = load_digits()
X_train_dig, X_test_dig, y_train_dig, y_test_dig = train_test_split(
    digits.data, digits.target, test_size=0.2, random_state=42
)

# Compare three Naive Bayes variants
models = {
    'Gaussian NB': GaussianNB(),
    'Multinomial NB': MultinomialNB(),
    'Bernoulli NB': BernoulliNB()
}

print("Naive Bayes Model Comparison (Digits):")
print("-" * 50)
for name, model in models.items():
    model.fit(X_train_dig, y_train_dig)
    acc = model.score(X_test_dig, y_test_dig)
    print(f"  {name:18s}: {acc:.4f}")

## 13. Online Learning (Incremental Learning)

Naive Bayes supports online learning via `partial_fit`.
Useful for large-scale data or streaming data.

In [ ]:
# Online learning simulation
gnb_online = GaussianNB()

# Batch learning
batch_size = 50
n_batches = len(X_train) // batch_size

for i in range(n_batches):
    start = i * batch_size
    end = start + batch_size
    X_batch = X_train[start:end]
    y_batch = y_train[start:end]

    # Define classes in the first batch
    if i == 0:
        gnb_online.partial_fit(X_batch, y_batch, classes=np.unique(y_train))
    else:
        gnb_online.partial_fit(X_batch, y_batch)

print("Online Learning Results:")
print(f"  Number of batches: {n_batches}")
print(f"  Batch size: {batch_size}")
print(f"  Accuracy: {gnb_online.score(X_test, y_test):.4f}")

## 14. kNN vs Naive Bayes Comparison

In [ ]:
# Breast cancer data
cancer = load_breast_cancer()
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42
)

# Scaling
scaler = StandardScaler()
X_train_c_scaled = scaler.fit_transform(X_train_c)
X_test_c_scaled = scaler.transform(X_test_c)

# Model comparison
models = {
    'kNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'kNN (weighted)': KNeighborsClassifier(n_neighbors=5, weights='distance'),
    'Gaussian NB': GaussianNB()
}

print("kNN vs Naive Bayes Comparison (Breast Cancer):")
print("=" * 50)

for name, model in models.items():
    if 'kNN' in name:
        model.fit(X_train_c_scaled, y_train_c)
        acc = model.score(X_test_c_scaled, y_test_c)
    else:
        model.fit(X_train_c, y_train_c)
        acc = model.score(X_test_c, y_test_c)
    print(f"  {name:18s}: {acc:.4f}")

## 15. Simple Text Classification Example

In [ ]:
# Why: TF-IDF (Term Frequency-Inverse Document Frequency) downweights common words
# like "the" and upweights distinctive words. This gives MultinomialNB better
# signal than raw counts — common words would otherwise dominate class probabilities.
tfidf = TfidfVectorizer()
X_sentiment = tfidf.fit_transform(texts)

# Train Naive Bayes
mnb_sentiment = MultinomialNB()
mnb_sentiment.fit(X_sentiment, labels)

# Classify new texts
new_texts = [
    "This is a great movie",
    "I hate this film",
    "Excellent performance and story",
    "Terrible and disappointing"
]
X_new = tfidf.transform(new_texts)
predictions = mnb_sentiment.predict(X_new)
probabilities = mnb_sentiment.predict_proba(X_new)

print("Sentiment Classification Results:")
print("=" * 60)
for text, pred, prob in zip(new_texts, predictions, probabilities):
    sentiment = "Positive" if pred == 1 else "Negative"
    confidence = max(prob) * 100
    print(f"'{text}'")
    print(f"  -> {sentiment} (confidence: {confidence:.1f}%)\n")

## Summary

### kNN Summary

| Parameter | Description | Recommendation |
|-----------|-------------|----------------|
| **n_neighbors** | Number of neighbors (k) | Select via cross-validation |
| **weights** | Weight method | 'distance' recommended |
| **metric** | Distance metric | 'euclidean' default |
| **algorithm** | Search algorithm | 'auto' |

**Characteristics**:
- Lazy learning (no training time)
- Slow prediction time (O(n*d))
- Scaling required
- Performance degrades in high dimensions (curse of dimensionality)

### Naive Bayes Summary

| Type | Feature Type | Primary Use |
|------|-------------|-------------|
| **GaussianNB** | Continuous (normal distribution) | General classification |
| **MultinomialNB** | Count/frequency | Text classification |
| **BernoulliNB** | Binary (0/1) | Word presence/absence |

**Characteristics**:
- Very fast (training O(n*d), prediction O(d))
- Works well even with small data
- Effective for high-dimensional data
- Supports online learning
- Feature independence assumption (may be violated in practice)

### kNN vs Naive Bayes

| Property | kNN | Naive Bayes |
|----------|-----|-------------|
| **Training time** | O(1) | O(n*d) |
| **Prediction time** | O(n*d) | O(d) |
| **Memory** | High | Low |
| **Scaling** | Required | Not required |
| **High dimensions** | Weak | Strong |
| **Interpretability** | Intuitive | Probability-based |

### Next Steps
- Clustering (K-Means, DBSCAN)
- Dimensionality Reduction (PCA, t-SNE)
- Ensemble methods (Stacking, Voting)